# AuraGateway CUDA 12.9 base-pip installation executor inspection v2

Corrects v1's invalid hard-coded `wrapt` assumption. Selects the smallest exact-locked pure-Python wheel without script data, then tests base-pip `--prefix` and `--target` installation into separate pip-less virtual environments. No full closure, model, requests, benchmark trajectory, or qualification.


In [ ]:

from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.parse
import zipfile
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

NOTEBOOK_NAME = "auragateway-cu129-pip-prefix-inspect-v2"
INPUT_DIRECTORY_NAME = "auragateway_vllm_cu129_wheelhouse_v1"
OUTPUT_DIRECTORY_NAME = "auragateway_cu129_pip_prefix_inspection_evidence_v2"
OUTPUT_ZIP = Path(f"/kaggle/working/{OUTPUT_DIRECTORY_NAME}.zip")
EVIDENCE_ROOT = Path(f"/kaggle/working/{OUTPUT_DIRECTORY_NAME}")
PREFIX_VENV = Path("/kaggle/working/auragateway_cu129_prefix_probe_venv_v2")
TARGET_VENV = Path("/kaggle/working/auragateway_cu129_target_probe_venv_v2")
PROBE_LOCK = Path("/kaggle/working/auragateway_cu129_probe_lock_v2.txt")
MAX_EXCERPT = 30000
MAX_PROBE_WHEEL_BYTES = 8 * 1024 * 1024

EXPECTED_RESOLUTION_LOCK_SHA256 = (
    "1575538b0a412c9b030fc95ccada0f0527553b76f06ef6b2b72904e61c84870c"
)
EXPECTED_SHA256_MANIFEST_SHA256 = (
    "789fb23ab7d9c4f28dd909e808a53a65d692c0d7b43bc44da9e974817d771b8d"
)
EXPECTED_MATERIALIZATION_RECEIPT_SHA256 = (
    "52aa42b940dd606ab5685686ab893eb085efed2a7466989f654e870f4b360589"
)
EXPECTED_PACKAGE_COUNT = 176

SOURCE_VERIFIER = "auragateway-cu129-offline-verifier-v6"
SOURCE_VERIFIER_EVIDENCE_ZIP_SHA256 = (
    "852b6d497adf620eca90c3719fe6dee1e607528ace6ac76cdf24907c009ada1f"
)
FAILED_INSPECTION = "auragateway-cu129-pip-prefix-inspect-v1"
FAILED_INSPECTION_NOTEBOOK_SHA256 = (
    "3c073d7bf2385c9aab0972c05abcc2e410e606143f2ab505cc80cc59c803a545"
)
FAILED_INSPECTION_EVIDENCE_ZIP_SHA256 = (
    "b5f169d039544dcf304076b98613ff4b35525e1962daf9ff41f9ab275566c9e1"
)
FAILED_INSPECTION_EXECUTION_LOG_SHA256 = (
    "622c4728573b20b450553bedb43e20bd5b8558f285e676bf54226056204b57c4"
)
FAILED_INSPECTION_FIRST_DIVERGENCE = "probe_distribution_selection"
FAILED_INSPECTION_FAILURE_CODE = "HARDCODED_PROBE_DISTRIBUTION_ABSENT"

CONTROLLED_BOOTSTRAP = r"""
import site
import sys
import types
from pathlib import Path

expected = Path(sys.argv[1]).resolve()
payload = sys.argv[2]
payload_args = sys.argv[3:]

def sentinel(name):
    module = types.ModuleType(name)
    module.__file__ = f"<auragateway-suppressed-{name}>"
    return module

sys.modules["sitecustomize"] = sentinel("sitecustomize")
sys.modules["usercustomize"] = sentinel("usercustomize")
site.main()

target_site = (
    expected
    / "lib"
    / f"python{sys.version_info.major}.{sys.version_info.minor}"
    / "site-packages"
).resolve()

cleaned = []
for value in sys.path:
    if not value:
        cleaned.append(value)
        continue
    path = Path(value).resolve()
    is_target = path == target_site or target_site in path.parents
    is_package_path = any(
        part in {"site-packages", "dist-packages"}
        for part in path.parts
    )
    if is_package_path and not is_target:
        continue
    cleaned.append(value)

if str(target_site) not in cleaned:
    cleaned.insert(0, str(target_site))

sys.path[:] = cleaned
sys.argv = ["<auragateway-target-probe>", *payload_args]
exec(
    compile(payload, "<auragateway-target-probe>", "exec"),
    {"__name__": "__main__"},
)
"""

DISTRIBUTION_PROBE = r"""
import importlib.metadata
import json
import site
import sys
from pathlib import Path

expected = Path(sys.argv[1]).resolve()
distribution_name = sys.argv[2]
expected_version = sys.argv[3]

distribution = importlib.metadata.distribution(distribution_name)
distribution_root = Path(distribution.locate_file("")).resolve()
target_site = (
    expected
    / "lib"
    / f"python{sys.version_info.major}.{sys.version_info.minor}"
    / "site-packages"
).resolve()

external = []
for value in sys.path:
    if not value:
        continue
    path = Path(value).resolve()
    is_target = path == target_site or target_site in path.parents
    is_package_path = any(
        part in {"site-packages", "dist-packages"}
        for part in path.parts
    )
    if is_package_path and not is_target:
        external.append(value)

files = distribution.files or []
resolved_files = [
    str(Path(distribution.locate_file(item)).resolve())
    for item in files[:50]
]
all_files_within_prefix = all(
    Path(value).resolve() == expected
    or Path(value).resolve().is_relative_to(expected)
    for value in resolved_files
)

print(
    json.dumps(
        {
            "prefix_matches_expected": Path(sys.prefix).resolve() == expected,
            "base_prefix_differs": (
                Path(sys.base_prefix).resolve()
                != Path(sys.prefix).resolve()
            ),
            "distribution_name": distribution.metadata.get("Name"),
            "distribution_version": distribution.version,
            "expected_version": expected_version,
            "distribution_root": str(distribution_root),
            "distribution_within_prefix": (
                distribution_root == expected
                or distribution_root.is_relative_to(expected)
            ),
            "sampled_distribution_file_count": len(resolved_files),
            "sampled_distribution_files_within_prefix": (
                all_files_within_prefix
            ),
            "target_site_packages": str(target_site),
            "target_site_packages_present": target_site.is_dir(),
            "external_package_paths": external,
            "sitecustomize_origin": getattr(
                sys.modules.get("sitecustomize"),
                "__file__",
                None,
            ),
            "usercustomize_origin": getattr(
                sys.modules.get("usercustomize"),
                "__file__",
                None,
            ),
            "user_site_enabled": site.ENABLE_USER_SITE,
            "no_site_flag": sys.flags.no_site,
        },
        sort_keys=True,
    )
)
"""

DISTRIBUTION_SNAPSHOT_SCRIPT = r"""
import importlib.metadata
import json

items = sorted(
    (
        dist.metadata.get("Name", ""),
        dist.version,
        str(dist.locate_file("")),
    )
    for dist in importlib.metadata.distributions()
)
print(json.dumps(items))
"""


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def streaming_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sanitize(text: str | bytes | None) -> str:
    if text is None:
        return ""
    decoded = (
        text.decode("utf-8", errors="replace")
        if isinstance(text, bytes)
        else text
    )
    replacements = {
        "/kaggle/input": "<input>",
        "/kaggle/working": "<working>",
        os.environ.get("HOME", ""): "<home>",
    }
    for source, replacement in replacements.items():
        if source:
            decoded = decoded.replace(source, replacement)
    return decoded[-MAX_EXCERPT:]


def offline_environment(venv: Path | None = None) -> dict[str, str]:
    environment = {**os.environ}
    environment.pop("PYTHONPATH", None)
    environment.pop("PYTHONHOME", None)
    environment["PYTHONNOUSERSITE"] = "1"
    environment["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
    environment["PIP_NO_INDEX"] = "1"
    environment["PIP_NO_CACHE_DIR"] = "1"
    environment["HF_HUB_OFFLINE"] = "1"
    environment["TRANSFORMERS_OFFLINE"] = "1"
    if venv is not None:
        environment["VIRTUAL_ENV"] = str(venv)
        target_bin = str(venv / "bin")
        inherited = [
            item
            for item in environment.get("PATH", "").split(os.pathsep)
            if item and item != target_bin
        ]
        environment["PATH"] = os.pathsep.join([target_bin, *inherited])
    return environment


def run_command(
    role: str,
    argv: list[str],
    *,
    timeout: float,
    environment: dict[str, str] | None = None,
) -> dict[str, object]:
    started = time.monotonic()
    started_at = datetime.now(UTC).isoformat(timespec="seconds")
    env = offline_environment() if environment is None else environment
    try:
        result = subprocess.run(
            argv,
            check=False,
            capture_output=True,
            text=True,
            timeout=timeout,
            env=env,
        )
        return {
            "schema_version": "1.0.0",
            "command_role": role,
            "argv": [sanitize(item) for item in argv],
            "started_at": started_at,
            "duration_ms": int((time.monotonic() - started) * 1000),
            "returncode": result.returncode,
            "timed_out": False,
            "status": "PASSED" if result.returncode == 0 else "FAILED",
            "stdout_excerpt": sanitize(result.stdout),
            "stderr_excerpt": sanitize(result.stderr),
        }
    except subprocess.TimeoutExpired as exc:
        return {
            "schema_version": "1.0.0",
            "command_role": role,
            "argv": [sanitize(item) for item in argv],
            "started_at": started_at,
            "duration_ms": int((time.monotonic() - started) * 1000),
            "returncode": None,
            "timed_out": True,
            "status": "FAILED",
            "stdout_excerpt": sanitize(exc.stdout),
            "stderr_excerpt": sanitize(exc.stderr),
        }


def controlled_argv(
    venv: Path,
    payload: str,
    *args: str,
) -> list[str]:
    return [
        str(venv / "bin" / "python"),
        "-S",
        "-c",
        CONTROLLED_BOOTSTRAP,
        str(venv),
        payload,
        *args,
    ]


def parse_json_stdout(
    record: dict[str, object],
) -> dict[str, Any] | None:
    try:
        payload = json.loads(
            str(record["stdout_excerpt"]).strip()
        )
    except json.JSONDecodeError:
        return None
    return payload if isinstance(payload, dict) else None


def require_json_object(path: Path) -> dict[str, Any]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise RuntimeError(
            f"expected JSON object: {path.name}"
        )
    return payload


def normalize_name(value: str) -> str:
    return re.sub(r"[-_.]+", "-", value).lower()


def locate_input_root() -> Path:
    matches = tuple(
        path
        for path in Path("/kaggle/input").rglob(
            INPUT_DIRECTORY_NAME
        )
        if path.is_dir()
    )
    if len(matches) != 1:
        raise RuntimeError(
            "expected exactly one governed wheelhouse; "
            f"observed {len(matches)}"
        )
    return matches[0]


def wheel_characteristics(path: Path) -> dict[str, object]:
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
    lowered = [name.lower() for name in names]
    native_suffixes = (
        ".so",
        ".pyd",
        ".dll",
        ".dylib",
    )
    contains_native_binary = any(
        name.endswith(native_suffixes)
        for name in lowered
    )
    data_categories = sorted(
        {
            match.group("category")
            for name in names
            for match in [
                re.search(
                    r"\.data/(?P<category>[^/]+)/",
                    name,
                )
            ]
            if match is not None
        }
    )
    contains_scripts = "scripts" in data_categories
    dist_info_metadata = [
        name
        for name in names
        if name.endswith(".dist-info/METADATA")
    ]
    return {
        "contains_native_binary": contains_native_binary,
        "data_categories": data_categories,
        "contains_scripts": contains_scripts,
        "dist_info_metadata_count": len(
            dist_info_metadata
        ),
        "archive_member_count": len(names),
    }


def validate_input_and_select_probe(
    input_root: Path,
) -> dict[str, Any]:
    identities = {
        "resolution_lock.json": (
            EXPECTED_RESOLUTION_LOCK_SHA256
        ),
        "sha256_manifest.json": (
            EXPECTED_SHA256_MANIFEST_SHA256
        ),
        "materialization_receipt.json": (
            EXPECTED_MATERIALIZATION_RECEIPT_SHA256
        ),
    }
    for name, expected in identities.items():
        observed = streaming_sha256(input_root / name)
        if observed != expected:
            raise RuntimeError(
                f"{name} identity drifted"
            )

    resolution = require_json_object(
        input_root / "resolution_lock.json"
    )
    records = resolution.get("records")
    if (
        resolution.get("package_count")
        != EXPECTED_PACKAGE_COUNT
        or not isinstance(records, list)
        or len(records) != EXPECTED_PACKAGE_COUNT
    ):
        raise RuntimeError(
            "resolution lock package closure drifted"
        )

    candidates: list[dict[str, Any]] = []
    for record in records:
        if not isinstance(record, dict):
            continue
        name = record.get("normalized_name")
        version = record.get("version")
        filename = record.get("artifact_filename")
        expected_sha256 = record.get("sha256")
        if not all(
            isinstance(value, str)
            for value in (
                name,
                version,
                filename,
                expected_sha256,
            )
        ):
            continue

        decoded_filename = urllib.parse.unquote(
            filename
        )
        wheel_path = (
            input_root
            / "wheels"
            / decoded_filename
        )
        if (
            not wheel_path.is_file()
            or wheel_path.stat().st_size
            > MAX_PROBE_WHEEL_BYTES
        ):
            continue

        characteristics = wheel_characteristics(
            wheel_path
        )
        if (
            characteristics[
                "contains_native_binary"
            ]
            or characteristics["contains_scripts"]
            or characteristics[
                "dist_info_metadata_count"
            ]
            != 1
        ):
            continue

        candidates.append(
            {
                "normalized_name": normalize_name(
                    name
                ),
                "version": version,
                "wheel_filename": (
                    decoded_filename
                ),
                "wheel_path": wheel_path,
                "wheel_sha256": expected_sha256,
                "wheel_size_bytes": (
                    wheel_path.stat().st_size
                ),
                **characteristics,
            }
        )

    if not candidates:
        raise RuntimeError(
            "no bounded pure-Python probe wheel "
            "was found"
        )

    candidates.sort(
        key=lambda item: (
            int(item["wheel_size_bytes"]),
            str(item["normalized_name"]),
            str(item["wheel_filename"]),
        )
    )
    selected = candidates[0]

    observed_sha256 = streaming_sha256(
        selected["wheel_path"]
    )
    if observed_sha256 != selected["wheel_sha256"]:
        raise RuntimeError(
            "selected probe wheel identity drifted"
        )

    PROBE_LOCK.write_text(
        (
            f"{selected['normalized_name']}"
            f"=={selected['version']} "
            f"--hash=sha256:"
            f"{selected['wheel_sha256']}\n"
        ),
        encoding="utf-8",
    )

    return {
        "status": "PASSED",
        "package_count": EXPECTED_PACKAGE_COUNT,
        "candidate_count": len(candidates),
        "selection_policy": (
            "SMALLEST_LOCKED_PURE_PYTHON_WHEEL_"
            "WITHOUT_SCRIPT_DATA"
        ),
        "selected_distribution": (
            selected["normalized_name"]
        ),
        "selected_version": selected["version"],
        "selected_wheel": (
            selected["wheel_filename"]
        ),
        "selected_wheel_sha256": (
            selected["wheel_sha256"]
        ),
        "selected_wheel_size_bytes": (
            selected["wheel_size_bytes"]
        ),
        "selected_wheel_characteristics": {
            key: selected[key]
            for key in (
                "contains_native_binary",
                "data_categories",
                "contains_scripts",
                "dist_info_metadata_count",
                "archive_member_count",
            )
        },
        "resolution_lock_sha256": (
            EXPECTED_RESOLUTION_LOCK_SHA256
        ),
        "sha256_manifest_sha256": (
            EXPECTED_SHA256_MANIFEST_SHA256
        ),
        "materialization_receipt_sha256": (
            EXPECTED_MATERIALIZATION_RECEIPT_SHA256
        ),
        "network_access_requested": False,
        "model_requests_performed": 0,
        "qualification_claimed": False,
    }


def verify_distribution_record(
    role: str,
    record: dict[str, object],
    expected_name: str,
    expected_version: str,
) -> None:
    if record.get("status") != "PASSED":
        return
    if str(
        record.get("stderr_excerpt", "")
    ).strip():
        record["status"] = "FAILED"
        record["semantic_error"] = (
            "controlled target emitted stderr"
        )
        return

    payload = parse_json_stdout(record)
    expected_origins = {
        "sitecustomize_origin": (
            "<auragateway-suppressed-"
            "sitecustomize>"
        ),
        "usercustomize_origin": (
            "<auragateway-suppressed-"
            "usercustomize>"
        ),
    }
    observed_name = (
        normalize_name(
            str(
                payload.get(
                    "distribution_name",
                    "",
                )
            )
        )
        if payload is not None
        else ""
    )
    if (
        payload is None
        or payload.get(
            "prefix_matches_expected"
        )
        is not True
        or payload.get("base_prefix_differs")
        is not True
        or observed_name != expected_name
        or payload.get("distribution_version")
        != expected_version
        or payload.get(
            "distribution_within_prefix"
        )
        is not True
        or payload.get(
            "sampled_distribution_files_within_prefix"
        )
        is not True
        or payload.get(
            "sampled_distribution_file_count"
        )
        in {None, 0}
        or payload.get(
            "target_site_packages_present"
        )
        is not True
        or payload.get(
            "external_package_paths"
        )
        != []
        or payload.get("user_site_enabled")
        is not False
        or payload.get("no_site_flag") != 1
        or any(
            payload.get(key) != value
            for key, value
            in expected_origins.items()
        )
    ):
        record["status"] = "FAILED"
        record["semantic_error"] = (
            f"{role} target contract drifted"
        )


def write_record(
    name: str,
    payload: object,
) -> None:
    (EVIDENCE_ROOT / name).write_text(
        canonical_json(payload) + "\n",
        encoding="utf-8",
    )


def main() -> int:
    if (
        os.environ.get(
            "AURAGATEWAY_CUSTOMER_DATA_PRESENT"
        )
        == "1"
    ):
        raise RuntimeError(
            "customer data is prohibited"
        )

    for path in (
        EVIDENCE_ROOT,
        PREFIX_VENV,
        TARGET_VENV,
    ):
        if path.exists():
            shutil.rmtree(path)
    for path in (OUTPUT_ZIP, PROBE_LOCK):
        if path.exists():
            path.unlink()
    EVIDENCE_ROOT.mkdir(parents=True)

    records: list[dict[str, object]] = []

    try:
        input_root = locate_input_root()
        input_validation = (
            validate_input_and_select_probe(
                input_root
            )
        )
    except Exception as exc:
        input_root = None
        input_validation = {
            "status": "FAILED",
            "error_type": type(exc).__name__,
            "error_message": sanitize(str(exc)),
            "network_access_requested": False,
            "model_requests_performed": 0,
            "qualification_claimed": False,
        }

    write_record(
        "00_input_validation.json",
        input_validation,
    )

    if (
        input_validation["status"] == "PASSED"
        and input_root is not None
    ):
        records.append(
            run_command(
                "base_distribution_snapshot_before",
                [
                    sys.executable,
                    "-c",
                    DISTRIBUTION_SNAPSHOT_SCRIPT,
                ],
                timeout=60.0,
            )
        )

        for role, venv in (
            (
                "prefix_venv_create_without_pip",
                PREFIX_VENV,
            ),
            (
                "target_venv_create_without_pip",
                TARGET_VENV,
            ),
        ):
            records.append(
                run_command(
                    role,
                    [
                        sys.executable,
                        "-m",
                        "venv",
                        "--without-pip",
                        str(venv),
                    ],
                    timeout=180.0,
                )
            )

        prefix_create = next(
            record
            for record in records
            if record["command_role"]
            == "prefix_venv_create_without_pip"
        )
        target_create = next(
            record
            for record in records
            if record["command_role"]
            == "target_venv_create_without_pip"
        )

        if prefix_create["status"] == "PASSED":
            records.append(
                run_command(
                    "base_pip_prefix_install_probe",
                    [
                        sys.executable,
                        "-m",
                        "pip",
                        "--isolated",
                        "--disable-pip-version-check",
                        "install",
                        "--no-index",
                        "--no-cache-dir",
                        "--no-deps",
                        "--ignore-installed",
                        "--require-hashes",
                        "--find-links",
                        str(input_root / "wheels"),
                        "--prefix",
                        str(PREFIX_VENV),
                        "-r",
                        str(PROBE_LOCK),
                    ],
                    timeout=300.0,
                )
            )
        else:
            records.append(
                {
                    "command_role": (
                        "base_pip_prefix_install_probe"
                    ),
                    "status": (
                        "BLOCKED_BY_UPSTREAM_FAILURE"
                    ),
                    "blocked_by": (
                        "prefix_venv_create_without_pip"
                    ),
                }
            )

        prefix_install = next(
            record
            for record in records
            if record["command_role"]
            == "base_pip_prefix_install_probe"
        )
        if prefix_install["status"] == "PASSED":
            records.append(
                run_command(
                    "controlled_prefix_distribution_probe",
                    controlled_argv(
                        PREFIX_VENV,
                        DISTRIBUTION_PROBE,
                        str(PREFIX_VENV),
                        str(
                            input_validation[
                                "selected_distribution"
                            ]
                        ),
                        str(
                            input_validation[
                                "selected_version"
                            ]
                        ),
                    ),
                    timeout=60.0,
                    environment=offline_environment(
                        PREFIX_VENV
                    ),
                )
            )
        else:
            records.append(
                {
                    "command_role": (
                        "controlled_prefix_distribution_probe"
                    ),
                    "status": (
                        "BLOCKED_BY_UPSTREAM_FAILURE"
                    ),
                    "blocked_by": (
                        "base_pip_prefix_install_probe"
                    ),
                }
            )

        if target_create["status"] == "PASSED":
            target_site = (
                TARGET_VENV
                / "lib"
                / (
                    f"python{sys.version_info.major}."
                    f"{sys.version_info.minor}"
                )
                / "site-packages"
            )
            records.append(
                run_command(
                    "base_pip_target_directory_install_probe",
                    [
                        sys.executable,
                        "-m",
                        "pip",
                        "--isolated",
                        "--disable-pip-version-check",
                        "install",
                        "--no-index",
                        "--no-cache-dir",
                        "--no-deps",
                        "--ignore-installed",
                        "--require-hashes",
                        "--find-links",
                        str(input_root / "wheels"),
                        "--target",
                        str(target_site),
                        "-r",
                        str(PROBE_LOCK),
                    ],
                    timeout=300.0,
                )
            )
        else:
            records.append(
                {
                    "command_role": (
                        "base_pip_target_directory_install_probe"
                    ),
                    "status": (
                        "BLOCKED_BY_UPSTREAM_FAILURE"
                    ),
                    "blocked_by": (
                        "target_venv_create_without_pip"
                    ),
                }
            )

        target_install = next(
            record
            for record in records
            if record["command_role"]
            == "base_pip_target_directory_install_probe"
        )
        if target_install["status"] == "PASSED":
            records.append(
                run_command(
                    "controlled_target_directory_distribution_probe",
                    controlled_argv(
                        TARGET_VENV,
                        DISTRIBUTION_PROBE,
                        str(TARGET_VENV),
                        str(
                            input_validation[
                                "selected_distribution"
                            ]
                        ),
                        str(
                            input_validation[
                                "selected_version"
                            ]
                        ),
                    ),
                    timeout=60.0,
                    environment=offline_environment(
                        TARGET_VENV
                    ),
                )
            )
        else:
            records.append(
                {
                    "command_role": (
                        "controlled_target_directory_distribution_probe"
                    ),
                    "status": (
                        "BLOCKED_BY_UPSTREAM_FAILURE"
                    ),
                    "blocked_by": (
                        "base_pip_target_directory_install_probe"
                    ),
                }
            )

        expected_name = str(
            input_validation[
                "selected_distribution"
            ]
        )
        expected_version = str(
            input_validation["selected_version"]
        )
        for role in (
            "controlled_prefix_distribution_probe",
            "controlled_target_directory_distribution_probe",
        ):
            record = next(
                item
                for item in records
                if item["command_role"] == role
            )
            verify_distribution_record(
                role,
                record,
                expected_name,
                expected_version,
            )

        records.append(
            run_command(
                "base_distribution_snapshot_after",
                [
                    sys.executable,
                    "-c",
                    DISTRIBUTION_SNAPSHOT_SCRIPT,
                ],
                timeout=60.0,
            )
        )

    observed = {
        str(record["command_role"]): record
        for record in records
    }
    before = observed.get(
        "base_distribution_snapshot_before"
    )
    after = observed.get(
        "base_distribution_snapshot_after"
    )
    base_unchanged = (
        before is not None
        and after is not None
        and before.get("status") == "PASSED"
        and after.get("status") == "PASSED"
        and before.get("stdout_excerpt")
        == after.get("stdout_excerpt")
    )
    prefix_passed = (
        observed.get(
            "base_pip_prefix_install_probe",
            {},
        ).get("status")
        == "PASSED"
        and observed.get(
            "controlled_prefix_distribution_probe",
            {},
        ).get("status")
        == "PASSED"
    )
    target_passed = (
        observed.get(
            "base_pip_target_directory_install_probe",
            {},
        ).get("status")
        == "PASSED"
        and observed.get(
            "controlled_target_directory_distribution_probe",
            {},
        ).get("status")
        == "PASSED"
    )

    if prefix_passed and base_unchanged:
        disposition = (
            "BASE_PIP_PREFIX_INSTALL_CONFIRMED"
        )
        selected_executor = "BASE_PIP_PREFIX"
        next_gate = (
            "build_cu129_offline_runtime_"
            "compatibility_verifier_v7"
        )
    elif target_passed and base_unchanged:
        disposition = (
            "BASE_PIP_TARGET_DIRECTORY_INSTALL_CONFIRMED"
        )
        selected_executor = (
            "BASE_PIP_TARGET_DIRECTORY"
        )
        next_gate = (
            "build_cu129_offline_runtime_"
            "compatibility_verifier_v7"
        )
    else:
        disposition = (
            "NO_SAFE_BASE_PIP_INSTALL_EXECUTOR_CONFIRMED"
        )
        selected_executor = None
        next_gate = (
            "inspect_offline_wheel_installation_executor"
        )

    summary = {
        "schema_version": "1.0.0",
        "diagnostic_id": NOTEBOOK_NAME,
        "captured_at": datetime.now(
            UTC
        ).isoformat(timespec="seconds"),
        "inspection_status": "COMPLETED",
        "disposition": disposition,
        "selected_installation_executor": (
            selected_executor
        ),
        "next_gate": next_gate,
        "source_verifier": SOURCE_VERIFIER,
        "source_verifier_evidence_zip_sha256": (
            SOURCE_VERIFIER_EVIDENCE_ZIP_SHA256
        ),
        "failed_inspection": FAILED_INSPECTION,
        "failed_inspection_notebook_sha256": (
            FAILED_INSPECTION_NOTEBOOK_SHA256
        ),
        "failed_inspection_evidence_zip_sha256": (
            FAILED_INSPECTION_EVIDENCE_ZIP_SHA256
        ),
        "failed_inspection_execution_log_sha256": (
            FAILED_INSPECTION_EXECUTION_LOG_SHA256
        ),
        "failed_inspection_first_divergence": (
            FAILED_INSPECTION_FIRST_DIVERGENCE
        ),
        "failed_inspection_failure_code": (
            FAILED_INSPECTION_FAILURE_CODE
        ),
        "prefix_install_confirmed": prefix_passed,
        "target_directory_install_confirmed": (
            target_passed
        ),
        "base_distribution_metadata_unchanged": (
            base_unchanged
        ),
        "full_closure_installation_performed": False,
        "probe_distribution_installed": (
            str(
                input_validation.get(
                    "selected_distribution",
                    "",
                )
            )
            if input_validation["status"]
            == "PASSED"
            else None
        ),
        "probe_distribution_version": (
            str(
                input_validation.get(
                    "selected_version",
                    "",
                )
            )
            if input_validation["status"]
            == "PASSED"
            else None
        ),
        "network_access_requested": False,
        "model_loaded": False,
        "model_requests_performed": 0,
        "benchmark_trajectory_requests_performed": 0,
        "qualification_claimed": False,
        "credentials_used": False,
        "customer_data_used": False,
        "external_spend": 0,
    }

    for index, record in enumerate(
        records,
        start=1,
    ):
        role = str(record["command_role"])
        write_record(
            f"10_{index:02d}_{role}.json",
            record,
        )
    write_record("90_summary.json", summary)

    payloads = tuple(
        sorted(
            EVIDENCE_ROOT.iterdir(),
            key=lambda item: item.name,
        )
    )
    manifest = {
        "schema_version": "1.0.0",
        "entries": [
            {
                "path": path.name,
                "sha256": streaming_sha256(path),
                "size_bytes": path.stat().st_size,
            }
            for path in payloads
        ],
    }
    write_record(
        "99_evidence_sha256.json",
        manifest,
    )

    with zipfile.ZipFile(
        OUTPUT_ZIP,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as archive:
        for path in sorted(
            EVIDENCE_ROOT.iterdir(),
            key=lambda item: item.name,
        ):
            archive.write(
                path,
                arcname=path.name,
            )

    print(f"artifact={OUTPUT_ZIP}")
    print(
        f"size_bytes={OUTPUT_ZIP.stat().st_size}"
    )
    print(
        f"sha256={streaming_sha256(OUTPUT_ZIP)}"
    )
    print(
        "inspection_status="
        f"{summary['inspection_status']}"
    )
    print(
        f"disposition={summary['disposition']}"
    )
    print(
        "selected_installation_executor="
        f"{summary['selected_installation_executor']}"
    )
    print(
        "selected_probe_distribution="
        f"{summary['probe_distribution_installed']}"
    )
    print(
        "prefix_install_confirmed="
        f"{str(prefix_passed).lower()}"
    )
    print(
        "target_directory_install_confirmed="
        f"{str(target_passed).lower()}"
    )
    print(
        "base_distribution_metadata_unchanged="
        f"{str(base_unchanged).lower()}"
    )
    print(f"next_gate={summary['next_gate']}")
    print(
        "full_closure_installation_performed=false"
    )
    print("model_requests_performed=0")
    print("qualification_claimed=false")
    print("upload_only_this_file=true")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
